# MVCBM — diagnosis apendisitis pediatrik (training di Kaggle)

Notebook ini hanya memanggil fungsi dari file `.py` di repo `pediatric-kumadai`. Kode diedit secara lokal, di-push ke GitHub, lalu dijalankan di sini.

Sebelum *Run All*:
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → On** (untuk `git clone`).
3. **Add Input** → dataset Kaggle (Private) berisi folder `preprocessed_400/` dan `dicts_diagnosis/imputed/final/` (`app_data_train`, `app_data_test`) hasil `prepare_data.py`.
4. Jika repo GitHub private: **Add-ons → Secrets** → tambahkan `GITHUB_TOKEN` (personal access token dengan akses baca repo).

Output (log, checkpoint, kurva ROC/PR, ringkasan metrik) tersimpan di `/kaggle/working/runs/`.

## 1. Ambil kode dari GitHub

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/akhyaruu/pediatric-kumadai.git"
BRANCH = "main"
REPO_DIR = "/tmp/pediatric-kumadai"  # outside /kaggle/working, so the code is not saved as notebook output

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repository or no secret attached
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", auth_url, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, auth_url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)  # no token on disk
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=commit %h  %ci  %s"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

Cloning into '/tmp/pediatric-kumadai'...


commit 82ece67  2026-11-17 22:03:43 +0900  Merge branch 'main' of https://github.com/akhyaruu/pediatric-kumadai


## 2. Paket yang belum ada di image Kaggle

In [2]:
import importlib.util

missing = [pip_name for module, pip_name in [("progressbar", "progressbar2")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("missing packages installed:", missing or "none")

missing packages installed: none


## 3. Cek GPU, data, dan bobot ResNet-18

In [3]:
import kaggle_setup

kaggle_setup.print_environment()
DATA = kaggle_setup.find_data()
for key, path in DATA.items():
    print(f"{key:15s} {path}")
kaggle_setup.ensure_resnet_weights()

python 3.13.16 (/usr/bin/python3)
torch 2.11.0+cu130, CUDA build 13.0
GPU: Tesla T4, 14.6 GiB, compute capability 7.5
images          /kaggle/input/datasets/akhyaruu/pediatric/data_app/preprocessed_400
dict_file       /kaggle/input/datasets/akhyaruu/pediatric/data_app/dicts_diagnosis/imputed/final/app_data_train
dict_file_test  /kaggle/input/datasets/akhyaruu/pediatric/data_app/dicts_diagnosis/imputed/final/app_data_test
test_images     /kaggle/input/datasets/akhyaruu/pediatric/data_app/preprocessed_400
ResNet-18 weights: /tmp/pediatric-kumadai/pretrained_models/resnet/resnet18-5c106cde.pth


'/tmp/pediatric-kumadai/pretrained_models/resnet/resnet18-5c106cde.pth'

## 4. Training + testing MVCBM

`CONFIGS`: setiap config dijalankan untuk setiap seed di `SEEDS`, output ke `/kaggle/working/runs/<nama config>/seed<N>`. Kedua config memakai dictionary yang sama (5 konsep); config 2 konsep hanya memakai 2 konsep pertama, jadi perbandingannya adil. Paper melaporkan rata-rata beberapa seed. `OVERRIDES` dapat dipakai untuk mengubah isi config tanpa mengedit file, mis. `{"c_epochs": 2, "t_epochs": 2}` untuk uji cepat.

In [ ]:
import importlib

import train_mvcbm

importlib.reload(train_mvcbm)  # picks up a new version of the code after re-running step 1
importlib.reload(kaggle_setup)

CONFIGS = [
    "configs/mvcbm_seq_lstm_diag.yaml",  # 2 concepts
    "configs/mvcbm_seq_lstm_diag_5c.yaml",  # 5 concepts
]
SEEDS = [42]
OVERRIDES = {}

results = {}  # results[run][seed] = test metrics of the last epoch
for config_path in CONFIGS:
    run = os.path.splitext(os.path.basename(config_path))[0]
    results.setdefault(run, {})
    for seed in SEEDS:
        overrides = {**kaggle_setup.kaggle_overrides(DATA, run_name=f"{run}/seed{seed}"), **OVERRIDES}
        config, results[run][seed] = train_mvcbm.run(config_path, seed=seed, overrides=overrides)
        print(f"\n{run}, seed {seed}: outputs in {config['log_directory']}\n")

## 5. Ringkasan metrik test (epoch terakhir)

Konsep (urutan `CONCEPTS` di `prepare_data.py`): `c0` = Appendix_on_US, `c1` = Free_Fluids, `c2` = Appendix_Wall_Layers, `c3` = Target_Sign, `c4` = Surrounding_Tissue_Reaction. Tabel pertama membandingkan config (rata-rata dan, untuk lebih dari satu seed, standar deviasi); di bawahnya detail per seed.

In [ ]:
from IPython.display import display

comparison = kaggle_setup.compare_runs(results)
comparison.to_csv("/kaggle/working/runs/comparison.csv")
display(comparison)

for run, results_per_seed in results.items():
    summary = kaggle_setup.summarize(results_per_seed)
    summary.to_csv(f"/kaggle/working/runs/{run}/summary.csv")
    print(run)
    display(summary)